# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. 
We will print the top-level record sets (`@id`), list the fields within each, and show their associated columns by `@id`.

In [ ]:
# Explore available record sets and fields by @id
record_set_objs = list(dataset.record_sets())
if len(record_set_objs) == 0:
    print("No top-level record sets found in the metadata.")
else:
    print("Available Record Sets (@id):\n")
    for rs in record_set_objs:
        print(f"- {rs.id} (name: {rs.name})")
        if hasattr(rs, "fields"):
            for f in rs.fields:
                print(f"    - Field @id: {f.id}, name: {f.name}")
                if hasattr(f, "columns"):
                    for c in f.columns:
                        print(f"        - Column @id: {c.id}, name: {c.name}")
        print("")

# For reference, also show the first few record sets in short form
print("\nSummary:")
for rs in record_set_objs:
    print(f"- {rs.id} (fields: {[f.id for f in getattr(rs,'fields',[])][:3]} ...)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

If multiple record sets are available, we will extract data from each into its own DataFrame using the `@id` as the key.

In [ ]:
# Build a list of record set @id's
record_set_ids = [rs.id for rs in dataset.record_sets()]

dataframes = {}
for record_set_id in record_set_ids:
    # Fetch records using the record set @id
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(dataframes[record_set_id])} records for record set @id '{record_set_id}'.")
        else:
            print(f"No records found for record set @id '{record_set_id}'.")
    except Exception as e:
        print(f"Error loading from record set @id '{record_set_id}':", e)

# Preview the first record set (if any)
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nField names in record set @id '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# EDA: Filtering, normalization, grouping based on available fields
import numpy as np
import warnings
warnings.filterwarnings('ignore')

if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    print(f"Shape of DataFrame for record set '{main_record_set_id}':", df.shape)

    # Attempt to infer numeric columns and choose one for demonstration
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    # If no numeric columns found, try to parse any column with likely numeric content
    if not numeric_candidates:
        # Try converting columns to numeric, coerce errors, and see which ones aren't all NaN
        for col in df.columns:
            converted = pd.to_numeric(df[col], errors='coerce')
            if converted.notnull().sum() > 0:
                numeric_candidates.append(col)

    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]  # Choose the first available numeric field
        print(f"Using numeric field for filtering: {numeric_field_id}")
        # Coerce to numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = np.nanquantile(df[numeric_field_id], 0.5)  # Median as demo threshold

        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        if filtered_df[numeric_field_id].std() != 0 and filtered_df[numeric_field_id].notnull().sum() > 0:
            filtered_df[f"{numeric_field_id}_normalized"] = (
                (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
                filtered_df[numeric_field_id].std()
            )
            print(f"Normalized '{numeric_field_id}' for filtered records:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        else:
            print(f"Standard deviation is zero or no valid numeric data for {numeric_field_id}.")

        # Attempt to group by a categorical field
        cat_candidates = [c for c in df.columns if df[c].dtype == 'object']
        # Avoid grouping by all-unique fields
        group_field_id = None
        for c in cat_candidates:
            if df[c].nunique() < len(df) and df[c].nunique() > 1:
                group_field_id = c
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean of '{numeric_field_id}' grouped by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields found to demonstrate filtering/normalization.")
else:
    print("No main record set DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. In this example, we will plot histograms for numeric fields and a bar plot for the top grouping.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_candidates:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True, color='skyblue')
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8, 4))
        sns.barplot(x=grouped_df[group_field_id], y=grouped_df[numeric_field_id])
        plt.title(f"Mean '{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
In this notebook, we demonstrated loading a FAIR dataset described with a Croissant schema using the `mlcroissant` library.

- We explored available record sets and fields using their `@id`s for precise reference.
- Data was loaded and converted into pandas DataFrames for analysis.
- Basic EDA operations were performed including filtering, normalization, grouping, and visualization of numeric and categorical fields, using dynamic selection based on dataset content.

This workflow provides a reproducible, schema-driven approach for dataset exploration and supports transparent referencing by croissant `@id` throughout your workflow.